# Phase 2 — EDA: Exploratory Data Analysis

**Objective:**
- Deep exploratory analysis of the training dataset.
- NLP preprocessing pipeline walkthrough.
- Train/Validation split creation.
- Verify split consistency and class stratification.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from data_preprocessing import clean_text, preprocess_dataframe

%matplotlib inline
ROOT = Path('..')
print('Ready.')

## 1. Load & Inspect Training Data

In [ ]:
df = pd.read_csv(ROOT / 'data' / 'train.csv')
print(f'Training samples: {len(df):,}')
print(f'Categories      : {df["category"].nunique()}')
df.head(10)

## 2. NLP Preprocessing Pipeline

In [ ]:
# Test the preprocessing on sample texts
sample_texts = [
    "I CAN'T withdraw money from the ATM!!!",
    "Why was I charged a fee???    Please help.",
    "My card hasn't arrived yet.  It's been 3 weeks!",
    "I didn't authorise this payment.",
    "URGENT: money debited but not received.",
]

print('Preprocessing demonstrations:')
for text in sample_texts:
    cleaned = clean_text(text)
    print(f'  Original : {text}')
    print(f'  Cleaned  : {cleaned}')
    print()

In [ ]:
# Verify that negation words are preserved
negation_test = "I didn't authorise this. I never agreed. Can't do that. No way."
cleaned = clean_text(negation_test)
print(f'Input  : {negation_test}')
print(f'Output : {cleaned}')

preserved = all(w in cleaned for w in ['not', 'never', 'no', "can't"])
print(f'Negation words preserved: {preserved}')

## 3. Preprocessing the Training Dataset

In [ ]:
# Check if preprocessed files exist from previous run
train_split_path = ROOT / 'data' / 'train_split.csv'
val_split_path   = ROOT / 'data' / 'val_split.csv'

if train_split_path.exists() and val_split_path.exists():
    df_train = pd.read_csv(train_split_path)
    df_val   = pd.read_csv(val_split_path)
    print(f'Loaded existing splits: train={len(df_train):,}, val={len(df_val):,}')
else:
    print('Running preprocessing and split...')
    os.chdir('..')
    from data_preprocessing import run as run_preprocessing
    run_preprocessing()
    df_train = pd.read_csv(train_split_path)
    df_val   = pd.read_csv(val_split_path)

df_train.head()

## 4. Word Count Analysis After Preprocessing

In [ ]:
df_train['word_count'] = df_train['clean_text'].str.split().str.len()

print('Word count statistics (training split):')
print(df_train['word_count'].describe())

# Recommend max_len
p95 = np.percentile(df_train['word_count'], 95)
p99 = np.percentile(df_train['word_count'], 99)
print(f'\n95th percentile word count : {p95:.0f}')
print(f'99th percentile word count : {p99:.0f}')
print(f'Recommended max_len        : 64 (covers >{(df_train["word_count"] <= 64).mean():.0%} of texts)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_train['word_count'], bins=40, color='#06d6a0', alpha=0.8, edgecolor='none')
axes[0].axvline(64, color='#ef476f', linestyle='--', lw=2, label='max_len=64')
axes[0].set_title('Word Count Distribution (Training Split)')
axes[0].set_xlabel('Word Count')
axes[0].legend()

# Category distribution in train vs val
train_cats = df_train['category'].value_counts().sort_index()
val_cats   = df_val['category'].value_counts().sort_index()
ratio = val_cats / (train_cats + val_cats)
axes[1].hist(ratio.dropna(), bins=20, color='#118ab2', alpha=0.8, edgecolor='none')
axes[1].axvline(0.2, color='#ef476f', linestyle='--', lw=2, label='Target 20%')
axes[1].set_title('Val Fraction per Category (Stratification Check)')
axes[1].set_xlabel('Val Fraction')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Most Common Words per Category

In [ ]:
# Show top 10 words for a few representative categories
sample_cats = ['cash_withdrawal_charge_reversed', 'card_arrival', 'pending_transfer']

for cat in sample_cats:
    subset = df_train[df_train['category'] == cat]['clean_text']
    if subset.empty:
        continue
    words = Counter()
    for text in subset:
        words.update(str(text).split())
    print(f"\nCategory: {cat}  (n={len(subset)})")
    print(f"  Top 10 words: {[w for w,_ in words.most_common(10)]}")

## Summary

| Property | Value |
|---------|-------|
| Training split size | ~8,002 |
| Validation split size | ~2,001 |
| Split ratio | 80/20 stratified |
| Negation words preserved | ✅ |
| Recommended max_len | 64 tokens |

**Next step:** Phase 3 — Traditional ML Baseline